# FE pipeline w/ placeholders for Modeling
* run with features_pipeline environment
* creating sample pipeline, assuming the data is geospatially aligned (in the examples they will not be)

In [1]:
# FEAT. AGG.
from fe_2d import *
from fe_3dep import *
from fe_als import *
from fe_rap import *

c:\Users\seani\miniforge3\envs\features_pipeline\Lib\site-packages\pyproj\network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


# RAP

In [2]:
def get_feature_complete_rap():
    ee.Authenticate()
    ee.Initialize(project="shrub-488520") # this might cause issues with the pipeline if we need to login.

    rap_image = ee.ImageCollection("projects/rap-data-365417/assets/vegetation-cover-10m").first()
    actual_bands = list(set(rap_image.bandNames().getInfo()) - {"SHR"})

    veg_features = extract_coarse_vegetation_prior(
        veg_image=rap_image,
        shrub_band_name='SHR', 
        comp_band_names=actual_bands, # Perennial and Annual grasses
        context_radius=500
    )

    export_region = ee.Geometry.Rectangle([-105.6, 40.35, -105.55, 40.4])
    save_as_tif(veg_features, export_region, "pipeline_processed/rap_pipeline_test.tif")

# ALS 

In [3]:
from fe_als import *

def get_feature_complete_als():
    """
    als test
    """
    process_las_roughness("pipeline_unprocessed/lid_1.laz", "pipeline_partially_processed/als_pipeline_test_roughness.laz")
    process_las_knn_metrics("pipeline_partially_processed/als_pipeline_test_roughness.laz", "pipeline_partially_processed/als_pipeline_test_knn_metrics.laz")
    calculate_height_above_ground("pipeline_partially_processed/als_pipeline_test_knn_metrics.laz", "pipeline_partially_processed/als_pipeline_test_hag.laz")
    calculate_surface_angle("pipeline_partially_processed/als_pipeline_test_hag.laz", "pipeline_partially_processed/als_pipeline_test_angle.laz")
    segment_trees_dbscan("pipeline_partially_processed/als_pipeline_test_angle.laz", "pipeline_partially_processed/als_pipeline_test_dbscan.laz")

    # requires HAG --> generates tifs
    create_height_rasters("pipeline_partially_processed/als_pipeline_test_dbscan.laz", "pipeline_processed/als_pipeline_test_knn_metrics")

# 3DEP

In [4]:
from fe_3dep import *

def get_feature_complete_3dep():
    ee.Authenticate()
    ee.Initialize(project='shrub-488520')

    img_3dep = ee.ImageCollection("USGS/3DEP/10m_collection").first()
    feat_names_3dep, feats_3dep = extract_all_terrain_features(img_3dep) # access individual bands with .select([bands])

    export_region = ee.Geometry.Rectangle([162.2, 5.2, 162.25, 5.25])
    save_as_tif(feats_3dep, export_region, "pipeline_processed/3dep_pipeline_test.tif")

In [5]:
get_feature_complete_rap()
out_files_rap = ["pipeline_processed/rap_pipeline_test.tif"]

get_feature_complete_als()
feature_complete_suffixes = ["_CHM_Max", "_Mean", "_Distance_to_Tall_Canopy", "_Local_Relief"]
out_files_als = [f"pipeline_processed/als_pipeline_test_knn_metrics{suffix}.tif" for suffix in feature_complete_suffixes]

get_feature_complete_3dep()
out_files_3dep = ["pipeline_processed/3dep_pipeline_test.tif"]

out_files = out_files_rap + out_files_als + out_files_3dep
out_files

['pipeline_processed/rap_pipeline_test.tif',
 'pipeline_processed/als_pipeline_test_knn_metrics_CHM_Max.tif',
 'pipeline_processed/als_pipeline_test_knn_metrics_Mean.tif',
 'pipeline_processed/als_pipeline_test_knn_metrics_Distance_to_Tall_Canopy.tif',
 'pipeline_processed/als_pipeline_test_knn_metrics_Local_Relief.tif',
 'pipeline_processed/3dep_pipeline_test.tif']

# 2D FEng + NAIP

In [ ]:
fe_2d_feats = [
    # setup
    extract_color_channels, 
    get_all_bands,  
    # general keep
    get_granulometry_features,
    get_fractal_dimension_map,
    get_wavelet_features,
    fast_cpu_gabor,
    get_uniform_blur,
    get_fast_lbp_texture,
    get_ldp_feature,
    get_entropy_feature, 
    get_first_four_moments,
    # arbitrary kernel arg
    get_color_diff,
    # colors 
    get_ndvi, # input bands - red, NIR
    get_vari, # input bands - red, green, blue
]

skipping color channels for simplicity. need to label bands to ensure color channels are being accessed and not arbitrary bands

In [56]:
for out_file in out_files:
    comb_feats = list()
    with rasterio.open(out_file) as src:
        gen_feats = [
            get_granulometry_features,
            get_fractal_dimension_map,
            get_wavelet_features,
            fast_cpu_gabor,
            get_uniform_blur,
            get_fast_lbp_texture,
            get_ldp_feature,
            get_entropy_feature, 
            get_first_four_moments,
        ]

        all_feats = list()
        for band in range(1, src.count + 1):
            # TODO - 3dep and rap tifs are saving with all 0 values. need to look into why this is happening
            # print(np.mean(src.read(band) == 0)) 
            for get_feat in gen_feats:
                all_feats.append(get_feat(src.read(band)))

        comb_feats.extend(all_feats)


    # for feat in all_feats:
    #     dtype = type(feat)
    #     if dtype == np.ndarray:
    #         print(feat.shape)
    #     else:
    #         print(len(feat))
len(comb_feats)

c:\Users\seani\miniforge3\envs\features_pipeline\Lib\site-packages\skimage\feature\texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(


81

In [59]:
# TODO generate naip files and process their bands

In [57]:
comb_feats = [np.array(feat) for feat in comb_feats]
comb_feats = np.concat([feat.flatten() for feat in comb_feats])
len(comb_feats)

72728604

# process outline (TODO)
1. repeat this feature generation process. 
2. find a way to align images (e.g. take smaller chunks) across data sources  
3. get labels for relevant locations
4. train model
5. test model -- if you don't have time to generate a lot of data just test with the training data

## image sources
* 3dep - ee.Image (earth engine)
* als - .laz
* rap - ee.Image (earth engine)
* naip - find out in morning 